# Business Understanding

*Generated by DoML — Do Machine Learning*

In [ ]:
# Reproducibility — REPR-01 (non-negotiable per CLAUDE.md)
import os
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
import json
import re
from pathlib import Path
from IPython.display import Markdown, display

# Project root — REPR-02 (non-negotiable per CLAUDE.md)
# Never hardcode absolute paths — always resolve from PROJECT_ROOT
PROJECT_ROOT = Path(os.environ.get('PROJECT_ROOT', '/home/jovyan/work'))

# Load project config
with open(PROJECT_ROOT / '.planning' / 'config.json') as f:
    config = json.load(f)

# Load PROJECT.md narrative
project_md = (PROJECT_ROOT / '.planning' / 'PROJECT.md').read_text()

print(f"Project root: {PROJECT_ROOT}")
print(f"Problem type: {config.get('problem_type', 'unknown')}")
print(f"Language: {config.get('language', 'python')}")
print(f"Time factor: {config.get('time_factor', False)}")

## Business Context

In [ ]:
def extract_field(text, label):
    """Extract content after a bold label: **Label:** content (until next bold label or heading)"""
    pattern = rf'\*\*{re.escape(label)}:\*\*\s*(.*?)(?=\n\*\*|\n##|\Z)'
    match = re.search(pattern, text, re.DOTALL)
    return match.group(1).strip() if match else 'Not specified'

def extract_framing(text):
    """Extract the decision framing blockquote: > We are trying to ..."""
    match = re.search(r'>\s*(We are trying to.*?)(?=\n\n|\Z)', text, re.DOTALL)
    return match.group(1).strip() if match else 'Not specified'

business_question = extract_field(project_md, 'Business question')
stakeholder = extract_field(project_md, 'Stakeholder')
expected_outcome = extract_field(project_md, 'Expected outcome')
framing = extract_framing(project_md)

display(Markdown(f"""
**Business question:** {business_question}

**Stakeholder:** {stakeholder}

**Expected outcome:** {expected_outcome}

---

**Decision framing:**
> {framing}
"""))

## Data Inventory

DuckDB scan of `data/raw/` — shape, schema, sample rows, and null counts per file.

In [ ]:
import duckdb
import pandas as pd

data_dir = PROJECT_ROOT / 'data' / 'raw'

SUPPORTED = {'.csv', '.parquet', '.xlsx'}
files = sorted([f for f in data_dir.glob('**/*') if f.is_file() and f.suffix.lower() in SUPPORTED])

if not files:
    print(f"No supported files found in {data_dir}")
    print("Supported formats: CSV (.csv), Parquet (.parquet), Excel (.xlsx)")
else:
    for path in files:
        con = duckdb.connect()
        p = str(path)
        ext = path.suffix.lower()

        if ext == '.csv':
            schema_df = con.execute(f"DESCRIBE SELECT * FROM read_csv_auto('{p}')").df()
            row_count = con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{p}')").fetchone()[0]
            sample_df = con.execute(f"SELECT * FROM read_csv_auto('{p}') LIMIT 3").df()
        elif ext == '.parquet':
            schema_df = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{p}')").df()
            row_count = con.execute(f"SELECT COUNT(*) FROM read_parquet('{p}')").fetchone()[0]
            sample_df = con.execute(f"SELECT * FROM read_parquet('{p}') LIMIT 3").df()
        elif ext == '.xlsx':
            schema_df = con.execute(f"DESCRIBE SELECT * FROM read_xlsx('{p}')").df()
            row_count = con.execute(f"SELECT COUNT(*) FROM read_xlsx('{p}')").fetchone()[0]
            sample_df = con.execute(f"SELECT * FROM read_xlsx('{p}') LIMIT 3").df()

        # Null counts — dynamic query across all columns
        cols = schema_df['column_name'].tolist()
        null_exprs = ', '.join(
            f'SUM(CASE WHEN "{c}" IS NULL THEN 1 ELSE 0 END) AS "{c}"'
            for c in cols
        )
        if ext == '.csv':
            null_df = con.execute(f"SELECT {null_exprs} FROM read_csv_auto('{p}')").df()
        elif ext == '.parquet':
            null_df = con.execute(f"SELECT {null_exprs} FROM read_parquet('{p}')").df()
        elif ext == '.xlsx':
            null_df = con.execute(f"SELECT {null_exprs} FROM read_xlsx('{p}')").df()
        con.close()

        null_summary = null_df.T.rename(columns={0: 'null_count'})
        null_summary['null_pct'] = (null_summary['null_count'] / row_count * 100).round(1)

        display(Markdown(f'### {path.name}'))
        print(f'Shape: {row_count:,} rows \u00d7 {len(cols)} columns')

        display(Markdown('**Schema:**'))
        display(schema_df[['column_name', 'column_type']].rename(
            columns={'column_name': 'Column', 'column_type': 'Type'}
        ))

        display(Markdown('**Sample rows (3):**'))
        display(sample_df)

        display(Markdown('**Null counts:**'))
        display(null_summary)
        print()

## ML Problem Type

In [ ]:
problem_type = config.get('problem_type', 'unknown')
time_factor = config.get('time_factor', False)
target_col = extract_field(project_md, 'Target variable')

display(Markdown(f"""
**Confirmed ML problem type:** `{problem_type}`

**Target variable:** {target_col}

**Time is a factor:** {'Yes' if time_factor else 'No'}
"""))

# Problem-type-specific notes
notes = {
    'regression': 'Predict a continuous numeric value. Evaluation: RMSE, MAE, R\u00b2.',
    'classification': 'Predict a category or binary outcome. Evaluation: accuracy, F1, AUC-ROC.',
    'clustering': 'Discover natural groups without labels. Evaluation: silhouette score, Davies-Bouldin.',
    'time_series': 'Ordered temporal observations \u2014 use TimeSeriesSplit, not random splits.',
    'dimensionality_reduction': 'Reduce feature space for visualization or downstream models.',
}
note = notes.get(problem_type, 'Problem type not recognized \u2014 verify config.json.')
print(f'Note: {note}')

## Dataset Provenance

In [ ]:
source = extract_field(project_md, 'Source')
description = extract_field(project_md, 'Description')
collection_period = extract_field(project_md, 'Collection period')
known_biases = extract_field(project_md, 'Known biases or limitations')

display(Markdown(f"""
**Source:** {source}

**Description:** {description}

**Collection period:** {collection_period}

**Known biases or limitations:** {known_biases}
"""))

## Caveats

*Assumptions and limitations for this analysis:*

- **Correlation is not causation** — all findings are observational unless a controlled experiment was run.
- This notebook documents context established during the kickoff interview. Assumptions may need revision as the data is explored.
- The Business Understanding phase does not modify raw data. Data in `data/raw/` is read-only (INFR-05).